# 07 · Transfer Learning and Modern Vision

**Part III — Computer vision** · Estimated time: ~120 min

## Learning objectives

- Explain why features learned on ImageNet transfer to other tasks, and decide between feature extraction and fine-tuning from the size and similarity of the target dataset.
- Load pretrained torchvision models with the `weights=` API, apply their preprocessing, and run them on new images.
- Adapt a pretrained ResNet to a new task by replacing its head, by fine-tuning with discriminative learning rates (parameter groups) and by gradual unfreezing; evaluate with test-time augmentation and inspect features with t-SNE.
- Implement Grad-CAM from scratch with forward and backward hooks and read its heatmaps critically.
- Build a compact Vision Transformer (patch embedding, class token, positional embeddings, encoder blocks) and compare it with a CNN on CIFAR-10.
- Implement a U-Net and train it for semantic segmentation with Dice and IoU metrics.
- Explain object-detection concepts (anchors, IoU, one- and two-stage detectors, DETR) and implement non-maximum suppression from scratch.
- Describe self-supervised and contrastive learning (SimCLR, MoCo, DINO, CLIP) and implement the NT-Xent loss.

## Prerequisites

- Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization (AdamW, schedules, augmentation)
- Notebook 05 · The PyTorch Workflow and Best Practices (hooks, `state_dict`, `Dataset`/`DataLoader`)
- Notebook 06 · Convolutional Neural Networks (convolutions, ResNets, feature maps, occlusion maps)

## Contents

1. Why transfer learning works
2. Pretrained models in torchvision
3. Feature extraction versus fine-tuning
4. Explaining predictions with Grad-CAM
5. Vision Transformers
6. Beyond classification: segmentation and detection
7. Self-supervised and contrastive learning

Exercises · Summary · Further reading

In [ ]:
import copy, math, os, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision import datasets
from torchvision.transforms import v2
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.datasets import load_sample_image
from sklearn.manifold import TSNE

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

USE_AMP = device.type == "cuda"          # mixed precision on GPU only (Notebook 14); no-op on CPU
PIN = device.type == "cuda"


def to_np(t):
    """Detach a tensor, move it to the CPU and hand it to NumPy (for plotting)."""
    return np.asarray(t.detach().cpu())


def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))


print("FAST_RUN =", FAST_RUN)

## 1. Why transfer learning works

### 1.1 Feature hierarchies are reusable

Notebook 06 · Convolutional Neural Networks ended with a look inside a trained CNN: the first layers had learned oriented edges and
colour contrasts, the middle layers textures and object parts, and only the last layers were specific to
the ten CIFAR-10 classes. This is not a peculiarity of CIFAR-10. Yosinski et al. (2014) measured it
systematically on ImageNet: features from the first layers of a network trained on one half of the
classes are *as good* for the other half as features trained on it directly, and transferability degrades
gradually with depth as the features become specific to the source classes (the drop is largest in the
last layers, and fine-tuning the transferred layers recovers most of it). Edges, textures and parts are
properties of natural images, not of any particular labelling of them.

**Transfer learning** exploits this. Instead of training a network from random initialisation on our
(usually small) dataset, we start from a network trained on a large **source** dataset, keep the
general-purpose early layers, and adapt the specific late layers to our **target** task. The pretrained
network has already spent its ImageNet budget (1.28 million images, hundreds of GPU-hours) learning what
natural images look like; we only need to teach it our labels. In practice this means

- much less data is needed: a few hundred images per class instead of tens of thousands;
- training is much faster: minutes instead of hours, because the optimiser starts near a good solution;
- accuracy is higher, often by a wide margin, on any task that involves natural images.

Since about 2015, essentially every practical computer-vision system starts from pretrained weights, and
the same is true for language (Notebook 10 · NLP with Pretrained Transformers (Hugging Face)), where the
source task is predicting the next word.

### 1.2 A decision matrix

How much of the network to retrain depends on two quantities: how much target data you have and how
similar it is to the source data.

| | target **similar** to source (natural photos) | target **different** (medical scans, satellite, sketches) |
|---|---|---|
| **small** dataset (hundreds–few thousand) | *Feature extraction*: freeze the backbone, train only a new linear head. Fine-tuning would overfit. | Hardest case. Extract features from *earlier* layers (more generic), train a small classifier; consider fine-tuning with strong regularisation or self-supervised pretraining on your own unlabelled data. |
| **large** dataset (tens of thousands+) | *Fine-tune* the whole network with a small learning rate; pretrained weights are a good initialisation. | Fine-tune the whole network, possibly with a larger learning rate; the initialisation still helps convergence even when the domain differs. |

Two more rules of thumb. First, the deeper the layer, the more task-specific it is, so if you fine-tune
only part of the network, unfreeze from the *end*. Second, use a learning rate one to two orders of
magnitude smaller than you would for training from scratch, at least for the pretrained layers: the goal
is to *nudge* good weights, not to replace them. Section 3 turns both rules into code.

> **Key idea.** A pretrained backbone is a general-purpose image feature extractor. Transfer learning
> replaces "learn features from scratch" by "reuse features and learn only the decision".

## 2. Pretrained models in torchvision

### 2.1 The `weights` API

`torchvision.models` ships dozens of architectures with weights trained on ImageNet-1k (and some on
larger datasets). Since torchvision 0.13 each model has a *weights enum* that bundles the checkpoint, its
preprocessing transform and metadata (accuracy, parameter count, class names). For ResNet-18:

- `ResNet18_Weights.IMAGENET1K_V1` is the original recipe (69.8 % top-1); `DEFAULT` is an alias for the
  best available weights and is what you should normally use. The old `pretrained=True` argument is
  deprecated and must not be used.
- `weights.transforms()` returns the exact preprocessing the weights were evaluated with: resize the
  shorter side to 256, centre-crop 224, convert to `float32` in $[0, 1]$, normalise with the ImageNet
  channel mean $(0.485, 0.456, 0.406)$ and standard deviation $(0.229, 0.224, 0.225)$. Feeding a
  pretrained model images normalised any other way silently degrades its accuracy.
- `weights.meta["categories"]` holds the 1,000 class names.

The checkpoint (45 MB for ResNet-18) is downloaded once into the torch hub cache
(`~/.cache/torch/hub/checkpoints/`, or `$TORCH_HOME/hub/checkpoints`) and reused afterwards. Because
this notebook must also run offline, the download is wrapped in `try/except`: if it fails, we fall back
to random initialisation with a loud warning. The rest of the notebook still runs, but predictions of the
"pretrained" model are then meaningless and the transfer-learning experiments lose their point.

In [ ]:
weights = ResNet18_Weights.DEFAULT
metrics = weights.meta.get("_metrics", {}).get("ImageNet-1K", {})
n_params_meta = weights.meta.get("num_params")
params_str = f"{n_params_meta:,}" if isinstance(n_params_meta, int) else "n/a"
print(f"weights: {weights} | params: {params_str} | "
      f"ImageNet top-1: {metrics.get('acc@1', 'n/a')} | top-5: {metrics.get('acc@5', 'n/a')}")

try:
    pretrained = resnet18(weights=weights)
    PRETRAINED = True
    print("ImageNet weights loaded (from the torch hub cache or freshly downloaded).")
except Exception as e:                                       # offline, proxy, disk full, ...
    PRETRAINED = False
    print(f"WARNING: could not obtain the pretrained weights ({type(e).__name__}: {str(e)[:80]}).")
    print("         Falling back to RANDOM initialisation: the notebook runs, but the model knows nothing.")
    pretrained = resnet18(weights=None)
pretrained = pretrained.to(device).eval()
pretrained_state = copy.deepcopy(pretrained.state_dict())    # kept for Section 3

preprocess = weights.transforms()
print("\npreprocessing:", preprocess)
IMAGENET_MEAN, IMAGENET_STD = tuple(preprocess.mean), tuple(preprocess.std)
categories = weights.meta["categories"]
print(f"\n{len(categories)} categories, e.g. {categories[:3]} ... {categories[-3:]}")

print("\nResNet-18 top-level modules:")
for name, child in pretrained.named_children():
    print(f"  {name:8s} {child.__class__.__name__:12s} {count_params(child):>10,} params")

The module list is the map we will use for the rest of the notebook: a stem (`conv1`, `bn1`, `relu`,
`maxpool`), four residual stages `layer1` to `layer4` with 64, 128, 256 and 512 channels, global average
pooling and a linear classifier `fc` with $512 \times 1000$ weights. Everything up to `avgpool` is the
**backbone**; `fc` is the **head**.

### 2.2 Running the classifier on new images

scikit-learn ships two photographs that need no download, a Chinese temple (`china.jpg`) and a flower
(`flower.jpg`). We preprocess them with `weights.transforms()` and print the five most probable ImageNet
classes. Note that the input to the transform is a `uint8` tensor of shape `(C, H, W)`; the output is
a normalised `float32` tensor of shape `(3, 224, 224)`, and the model wants a batch dimension in front.

In [ ]:
sample_images = {name: load_sample_image(name) for name in ["china.jpg", "flower.jpg"]}   # (H, W, 3) uint8


def to_chw_uint8(img_hwc):
    return torch.from_numpy(np.ascontiguousarray(img_hwc)).permute(2, 0, 1)


@torch.no_grad()
def predict_topk(model, img_hwc, k=5):
    """Top-k (class name, probability) of an ImageNet classifier on a (H, W, 3) uint8 image."""
    x = preprocess(to_chw_uint8(img_hwc))[None].to(device)               # (1, 3, 224, 224)
    probs = F.softmax(model(x).float(), dim=1)[0]
    top = probs.topk(k)
    return [(categories[i], p.item()) for p, i in zip(top.values, top.indices)]


fig, axes = plt.subplots(2, 2, figsize=(11, 7), gridspec_kw={"width_ratios": [1.3, 1]})
for row, (name, img) in enumerate(sample_images.items()):
    x = preprocess(to_chw_uint8(img))
    print(f"{name}: raw {img.shape} -> preprocessed {tuple(x.shape)}, mean {x.mean():.3f}, std {x.std():.3f}")
    top = predict_topk(pretrained, img)
    axes[row, 0].imshow(img); axes[row, 0].set_title(name); axes[row, 0].axis("off")
    axes[row, 1].barh([c[:22] for c, _ in top][::-1], [p for _, p in top][::-1])
    axes[row, 1].set(xlim=(0, 1), xlabel="probability", title="top-5 ImageNet classes")
plt.tight_layout(); plt.show()
if not PRETRAINED:
    print("(random weights: the predictions above are noise)")

With real weights the temple is recognised as a palace, monastery or similar building and the flower as
one of ImageNet's flower classes, with the remaining probability spread over visually related classes.
None of the 1,000 ImageNet categories is "Chinese temple", so the network answers with the closest
concepts it knows, which is a reminder that a classifier can only ever output one of the labels it was
trained on. Transfer learning is how we give it new ones.

## 3. Feature extraction versus fine-tuning

### 3.1 A target task: CIFAR-10 at 96 pixels

We reuse CIFAR-10 from Notebook 06 · Convolutional Neural Networks as the target dataset, for three reasons: it is small, we know what
a from-scratch CNN achieves on it (roughly 85-90 % after 12 epochs), and its classes are a subset of what
ImageNet covers (similar source and target: the top-left cell of the decision matrix). Two adaptations are
needed. The images must be normalised with the *ImageNet* statistics, not CIFAR's, because that is what
the pretrained weights expect. And they must be enlarged: ResNet-18 downsamples by 32x, so a
$32 \times 32$ input would reach `layer4` as a single pixel. We resize to $96 \times 96$: the $224$ the
network was trained at would mean processing 49x more pixels than the original thumbnail, whereas $96$
costs 9x and still leaves `layer4` with a $3 \times 3$ map.
Under `FAST_RUN` we use 1,024 training and 512 test images; the full run uses 10,000 and 2,000.
If CIFAR-10 cannot be downloaded (no internet), the next cell substitutes a synthetic CIFAR-like set of
coloured shapes with the same layout so that the notebook still runs; its numbers are then not comparable
to the ones quoted in the text.

In [ ]:
def _synthetic_cifar_like(n, seed):
    """Offline stand-in for CIFAR-10: 32x32 RGB images of ten colour-and-shape classes (random position,
    size, background and pixel noise). Same array layout as torchvision's CIFAR-10, so every cell below
    runs unchanged. Accuracies on it are NOT comparable to real CIFAR-10 results."""
    rng = np.random.default_rng(seed)
    labels = rng.integers(0, 10, size=n)
    colours = np.array([[210, 60, 60], [60, 190, 60], [60, 80, 220], [220, 200, 50], [200, 60, 200],
                        [60, 200, 210], [235, 140, 40], [130, 50, 220], [60, 220, 150], [170, 170, 170]], dtype=np.float32)
    yy, xx = np.mgrid[0:32, 0:32]
    images = np.empty((n, 32, 32, 3), dtype=np.uint8)
    for i in range(n):
        c = int(labels[i])
        img = np.ones((32, 32, 3), np.float32) * (rng.uniform(40, 140, size=3) + rng.normal(0, 4, size=3))
        cx, cy, r = rng.uniform(11, 21), rng.uniform(11, 21), rng.uniform(5, 10)
        dx, dy = xx - cx, yy - cy
        shape = c % 5                                       # 0 square, 1 disc, 2 triangle, 3 ring, 4 stripes
        if shape == 0:
            mask = (np.abs(dx) < r) & (np.abs(dy) < r)
        elif shape == 1:
            mask = dx ** 2 + dy ** 2 < r ** 2
        elif shape == 2:
            mask = (dy > -r) & (np.abs(dx) < (dy + r) * 0.6)
        elif shape == 3:
            d2 = dx ** 2 + dy ** 2
            mask = (d2 < r ** 2) & (d2 > (0.55 * r) ** 2)
        else:
            mask = (np.abs(dx) < r) & (np.abs(dy) < r) & (((yy - int(cy)) // 3) % 2 == 0)
        img[mask] = colours[c] * rng.uniform(0.85, 1.15, size=3)
        img += rng.normal(0, 10, img.shape)
        images[i] = np.clip(img, 0, 255).astype(np.uint8)
    return images, labels.tolist()


def load_cifar10_or_fallback(data_dir, n_train=10_000, n_test=2_000, seed=SEED):
    """Download CIFAR-10 if possible; when offline, return a synthetic CIFAR-like dataset so that the
    notebook still runs end to end. Returns (train, test, is_synthetic); both splits expose
    .data (N, 32, 32, 3) uint8, .targets (list of int) and .classes like torchvision's CIFAR10."""
    try:
        train = datasets.CIFAR10(str(data_dir), train=True, download=True)
        test = datasets.CIFAR10(str(data_dir), train=False, download=True)
        return train, test, False
    except Exception as e:                                   # no internet, blocked host, corrupt download
        from types import SimpleNamespace
        print(f"[offline fallback] CIFAR-10 could not be downloaded ({type(e).__name__}: {str(e)[:80]}).\n"
              f"  Using a SYNTHETIC CIFAR-like dataset of coloured shapes ({n_train:,} train / {n_test:,} test images)\n"
              f"  so that the rest of the notebook runs. Numbers below are NOT comparable to real CIFAR-10 results;\n"
              f"  re-run this cell with internet access (or place cifar-10-python.tar.gz in {data_dir}) for the real data.")
        classes = ["red square", "green disc", "blue triangle", "yellow ring", "magenta stripes",
                   "cyan square", "orange disc", "violet triangle", "mint ring", "grey stripes"]
        xtr, ytr = _synthetic_cifar_like(n_train, seed)
        xte, yte = _synthetic_cifar_like(n_test, seed + 1)
        return (SimpleNamespace(data=xtr, targets=ytr, classes=classes),
                SimpleNamespace(data=xte, targets=yte, classes=classes), True)


cifar_train_raw, cifar_test_raw, CIFAR_IS_SYNTHETIC = load_cifar10_or_fallback(DATA_DIR)
CLASSES = cifar_train_raw.classes
cifar_train_x = torch.from_numpy(cifar_train_raw.data).permute(0, 3, 1, 2).contiguous()     # uint8 (N, 3, 32, 32)
cifar_train_y = torch.tensor(cifar_train_raw.targets, dtype=torch.long)
cifar_test_x = torch.from_numpy(cifar_test_raw.data).permute(0, 3, 1, 2).contiguous()
cifar_test_y = torch.tensor(cifar_test_raw.targets, dtype=torch.long)


class TensorImageDataset(Dataset):
    """Images as one uint8 tensor (N, C, H, W) plus labels, with a per-sample transforms.v2 pipeline."""

    def __init__(self, images, labels, transform=None):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = self.images[i]
        return (self.transform(x) if self.transform is not None else x), self.labels[i]


def make_cifar_loaders(n_train, n_test, train_tf, eval_tf, batch_size=128, seed=SEED):
    """Loaders over a seeded random subset of CIFAR-10 train and the first n_test test images."""
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(cifar_train_y), generator=g)[:n_train]
    train_ds = TensorImageDataset(cifar_train_x[idx], cifar_train_y[idx], train_tf)
    test_ds = TensorImageDataset(cifar_test_x[:n_test], cifar_test_y[:n_test], eval_tf)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=PIN, drop_last=True),
            DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=0, pin_memory=PIN))


FT_SIZE = 96
ft_train_tf = v2.Compose([v2.Resize(FT_SIZE, antialias=True), v2.RandomCrop(FT_SIZE, padding=8),
                          v2.RandomHorizontalFlip(), v2.ToDtype(torch.float32, scale=True),
                          v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])
ft_eval_tf = v2.Compose([v2.Resize(FT_SIZE, antialias=True), v2.ToDtype(torch.float32, scale=True),
                         v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

N_FT_TRAIN = 1_024 if FAST_RUN else 10_000
N_FT_TEST = 512 if FAST_RUN else 2_000
EPOCHS_FT = 1 if FAST_RUN else 3
ft_train_loader, ft_test_loader = make_cifar_loaders(N_FT_TRAIN, N_FT_TEST, ft_train_tf, ft_eval_tf, batch_size=32)
xb, yb = next(iter(ft_train_loader))
print(f"fine-tuning data: {N_FT_TRAIN:,} train / {N_FT_TEST:,} test images at {FT_SIZE}x{FT_SIZE}, "
      f"{EPOCHS_FT} epoch(s); batch {tuple(xb.shape)}")

The training helpers below are those of Notebook 06 · Convolutional Neural Networks with three additions that this notebook needs:
`fit` accepts a list of **parameter groups** (for discriminative learning rates), can use AdamW (for the
Vision Transformer), and its schedule has a linear **warm-up** followed by cosine decay. When parameter
groups carry their own `lr`, `LambdaLR` scales each group's learning rate by the same multiplier, which is
exactly what we want.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scheduler, scaler):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(x)
            loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion=None, tta=False):
    """Mean loss and accuracy; with tta=True the logits of the image and its mirror image are averaged."""
    model.eval()
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(x).float()
            if tta:
                logits = 0.5 * (logits + model(torch.flip(x, dims=[3])).float())
        total_loss += criterion(logits, y).item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n


def fit(model, train_loader, test_loader, epochs, lr=1e-2, weight_decay=5e-4, optimizer="sgd",
        param_groups=None, criterion=None, warmup_frac=0.05, verbose=True):
    """Train with SGD+Nesterov or AdamW, linear warm-up then cosine decay (stepped per iteration)."""
    model.to(device)
    criterion = criterion if criterion is not None else nn.CrossEntropyLoss()
    params = param_groups if param_groups is not None else [p for p in model.parameters() if p.requires_grad]
    if optimizer == "sgd":
        opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay, nesterov=True)
    else:
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    total, warm = epochs * len(train_loader), int(warmup_frac * epochs * len(train_loader))

    def lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(total - warm, 1)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    hist = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [], "epoch_time": []}
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, criterion, sched, scaler)
        te_loss, te_acc = evaluate(model, test_loader, criterion)
        dt = time.time() - t0
        for k, v in zip(["train_loss", "train_acc", "test_loss", "test_acc", "epoch_time"], [tr_loss, tr_acc, te_loss, te_acc, dt]):
            hist[k].append(v)
        if verbose:
            print(f"epoch {epoch:2d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"test loss {te_loss:.4f} acc {te_acc:.3f} | {dt:5.1f}s")
    return hist


print("training helpers defined")

### 3.2 Feature extraction: a linear probe on frozen features

The cheapest form of transfer learning treats the pretrained backbone as a fixed function. We remove the
ImageNet head (replace `fc` by `nn.Identity()`, so the model outputs the 512-dimensional pooled features)
and train *only* a new linear classifier on top. Because nothing in the backbone changes, we can run every
image through it **once**, cache the 512 numbers, and train the classifier on the cached features for as
many epochs as we like at negligible cost. This is exactly equivalent to "freeze the backbone and train
the head" with the backbone in evaluation mode (so that BatchNorm uses its ImageNet running statistics),
and it is how a **linear probe**, the standard measure of representation quality in the self-supervised
learning literature (Section 7), is computed. The price is that data augmentation is impossible, since
every image is seen exactly once.

We also train the same linear classifier on features from a *randomly initialised* ResNet-18. This is
the control experiment: it tells us how much of the accuracy comes from the pretrained weights rather than
from the architecture.

In [ ]:
@torch.no_grad()
def extract_features(backbone, loader):
    """Run a loader through a headless backbone (eval mode); returns (features, labels) on the CPU."""
    backbone.eval()
    feats, labels = [], []
    for x, y in loader:
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            f = backbone(x.to(device, non_blocking=True))
        feats.append(f.float().cpu()); labels.append(y)
    return torch.cat(feats), torch.cat(labels)


def headless_resnet18(state_dict=None):
    """ResNet-18 whose fc is the identity: outputs the 512-d pooled features."""
    m = resnet18(weights=None)
    if state_dict is not None:
        m.load_state_dict(state_dict)
    m.fc = nn.Identity()
    return m.to(device)


def train_linear_probe(train_feats, train_labels, test_feats, test_labels, epochs=30, lr=1e-2):
    """Multinomial logistic regression on cached features, trained with our own fit()."""
    tr = DataLoader(TensorDataset(train_feats, train_labels), batch_size=128, shuffle=True)
    te = DataLoader(TensorDataset(test_feats, test_labels), batch_size=512)
    torch.manual_seed(SEED)
    clf = nn.Linear(train_feats.shape[1], 10)
    hist = fit(clf, tr, te, epochs=epochs, lr=lr, weight_decay=1e-4, optimizer="adamw", verbose=False)
    return clf, hist["test_acc"][-1]


results = {}                       # method -> (test accuracy, wall-clock seconds)

t0 = time.time()
backbone_pre = headless_resnet18(pretrained_state)
train_feats, train_labels = extract_features(backbone_pre, ft_train_loader)
test_feats, test_labels = extract_features(backbone_pre, ft_test_loader)
t_extract = time.time() - t0
print(f"features: train {tuple(train_feats.shape)}, test {tuple(test_feats.shape)} extracted in {t_extract:.1f}s")

t0 = time.time()
probe_pre, acc_probe = train_linear_probe(train_feats, train_labels, test_feats, test_labels)
results["linear probe (pretrained)"] = (acc_probe, t_extract + time.time() - t0)
print(f"linear probe on PRETRAINED features : test acc {100 * acc_probe:.1f}%")

t0 = time.time()
torch.manual_seed(SEED)
backbone_rand = headless_resnet18(None)
r_train_feats, r_train_labels = extract_features(backbone_rand, ft_train_loader)   # the loader shuffles:
r_test_feats, r_test_labels = extract_features(backbone_rand, ft_test_loader)      # keep the matching labels
_, acc_probe_rand = train_linear_probe(r_train_feats, r_train_labels, r_test_feats, r_test_labels)
results["linear probe (random init)"] = (acc_probe_rand, time.time() - t0)
print(f"linear probe on RANDOM features     : test acc {100 * acc_probe_rand:.1f}%")

With real ImageNet weights the linear probe reaches roughly 75-85 % on CIFAR-10 in the full run (somewhat
less under `FAST_RUN`, where the probe sees only 1,024 training images) although the backbone made a single
pass over the training images, whereas the same classifier on random-network features stays around
30-40 %. The 512 pretrained features already separate cats from dogs and ships from trucks reasonably
well; the random network's features are a fixed random projection of the pixels, only slightly better
than raw pixels.
(Note that the training augmentation was applied once during extraction, so each cached training feature
corresponds to one random crop; this is harmless.) Everything the probe cannot fix lives in the frozen
backbone: features tuned to 224-pixel ImageNet photographs are not ideal for blurry 96-pixel upsampled
thumbnails. Fine-tuning addresses exactly that.

### 3.3 Fine-tuning with discriminative learning rates

**Fine-tuning** updates the pretrained weights themselves. The whole network is trained on the target
data with a small learning rate, so that the weights move a little towards the new task without
forgetting what they know (the failure mode when the learning rate is too high is called **catastrophic
forgetting**: a few large steps destroy the ImageNet features, and the network ends up no better than one
trained from scratch on the small dataset).

Since deeper layers are more task-specific, it makes sense to let them move more. **Discriminative
learning rates** (Howard and Ruder, 2018, who introduced the term for language models) assign each layer
group its own learning rate, decreasing geometrically from the head towards the stem:
$\eta_{\text{layer}} = \eta_{\text{head}} / 2.6^{\,d}$, where $d$ is the depth counted from the head. The
new randomly initialised head gets the full learning rate, `layer4` gets $\eta / 2.6$, and so on down to
the stem, which barely moves. PyTorch implements this with **parameter groups**: an optimiser accepts a
list of dictionaries, each with its own `params` and hyperparameters.

We also train the identical architecture **from scratch** with the same budget (and a larger learning
rate, since random weights need bigger steps), the control that shows what the pretrained initialisation
is worth.

In [ ]:
def resnet18_for_cifar(pretrained_init=True, num_classes=10):
    """ResNet-18 with a fresh 10-class head, initialised from ImageNet weights or randomly."""
    m = resnet18(weights=None)
    if pretrained_init and PRETRAINED:
        m.load_state_dict(pretrained_state)
    m.fc = nn.Linear(m.fc.in_features, num_classes)           # new head (random init)
    return m


def discriminative_param_groups(model, base_lr, decay=2.6, only_trainable=False):
    """Parameter groups with lr = base_lr / decay**depth, depth counted from the head (fc = 0)."""
    stages = [["fc"], ["layer4"], ["layer3"], ["layer2"], ["conv1", "bn1", "layer1"]]
    groups = []
    for depth, names in enumerate(stages):
        params = [p for n in names for p in getattr(model, n).parameters()
                  if (p.requires_grad or not only_trainable)]
        if params:
            groups.append({"params": params, "lr": base_lr / decay ** depth, "name": "+".join(names)})
    return groups


torch.manual_seed(SEED)
ft_model = resnet18_for_cifar(pretrained_init=True)
groups = discriminative_param_groups(ft_model, base_lr=0.01)
for g in groups:
    print(f"{g['name']:18s} lr = {g['lr']:.5f}   ({sum(p.numel() for p in g['params']):>10,} params)")

print("\n--- fine-tuning from ImageNet weights")
t0 = time.time()
hist_ft = fit(ft_model, ft_train_loader, ft_test_loader, EPOCHS_FT, lr=0.01, param_groups=groups)
results["fine-tuning (discriminative LR)"] = (hist_ft["test_acc"][-1], time.time() - t0)

print("\n--- same architecture from scratch, same budget")
torch.manual_seed(SEED)
scratch_model = resnet18_for_cifar(pretrained_init=False)
t0 = time.time()
hist_scratch = fit(scratch_model, ft_train_loader, ft_test_loader, EPOCHS_FT, lr=0.05)
results["from scratch (same budget)"] = (hist_scratch["test_acc"][-1], time.time() - t0)

The contrast is the whole argument for transfer learning in one experiment. From ImageNet weights, the
network is typically already in the 60s or 70s after a single epoch on 1,024 images under `FAST_RUN`, and
in the high 80s to low 90s after three epochs on 10,000 images in the full run; from scratch, the same
architecture, data and budget give a fraction of that. The from-scratch ResNet-18 would need the full
50,000 images and tens of epochs to catch up, and on a dataset with a few hundred images per class it
never would.

### 3.4 Gradual unfreezing

A middle path between freezing everything and fine-tuning everything is to **unfreeze gradually**
(Howard and Ruder, 2018): first train only the head and the last stage, then also the stage before it,
and so on, one phase at a time. The early layers see gradients only once the later layers have already
adapted, so they are never hit by the large, noisy gradients that a randomly initialised head produces
in the first iterations. A parameter is frozen by setting `requires_grad = False`, which removes it from
the optimiser (`fit` collects only trainable parameters) and saves the backward computation through it.
Under `FAST_RUN` we run only the first phase; the full run uses three phases of one epoch each, the same
total budget as full fine-tuning.

> **Common pitfall.** `requires_grad_(False)` freezes a layer's *weights*, not its BatchNorm *running
> statistics*: as long as the model is in `train()` mode, every BatchNorm layer, frozen or not, keeps
> updating its running mean and variance with the target-domain batches. In gradual unfreezing this is
> harmless (every layer is eventually trained, and statistics adapted to the new images help), but when
> a backbone is meant to stay fixed, as in Section 3.2, keep it in `eval()` mode (or call `.eval()` on its
> BatchNorm modules after `model.train()`), otherwise the "frozen" network silently drifts with every batch,
> and with small batches the drift can cost several points of accuracy.

In [ ]:
def set_trainable(model, names):
    """Freeze every parameter except those of the listed top-level submodules."""
    for p in model.parameters():
        p.requires_grad_(False)
    for n in names:
        for p in getattr(model, n).parameters():
            p.requires_grad_(True)


phases = [["fc", "layer4"],
          ["fc", "layer4", "layer3"],
          ["fc", "layer4", "layer3", "layer2", "layer1", "bn1", "conv1"]][:EPOCHS_FT]

torch.manual_seed(SEED)
gu_model = resnet18_for_cifar(pretrained_init=True)
t0 = time.time()
for i, names in enumerate(phases):
    set_trainable(gu_model, names)
    print(f"phase {i + 1}: trainable = {' + '.join(names)} "
          f"({count_params(gu_model, trainable_only=True):,} of {count_params(gu_model):,} params)")
    groups = discriminative_param_groups(gu_model, base_lr=0.01, only_trainable=True)
    hist_gu = fit(gu_model, ft_train_loader, ft_test_loader, epochs=1, lr=0.01, param_groups=groups)
results["gradual unfreezing"] = (hist_gu["test_acc"][-1], time.time() - t0)

### 3.5 Comparison, test-time augmentation and a look at the features

The table collects the four strategies (plus the random-feature control) with their test accuracy and
wall-clock time. Then we try **test-time augmentation** (TTA): average the network's predictions over the
image and its horizontal mirror. TTA costs two forward passes per image and typically buys a few tenths of
a point; it is the cheapest accuracy improvement there is, and it is used in most competition entries.

In [ ]:
print(f"{'method':34s} {'test acc':>9s} {'time (s)':>9s}")
for name, (acc, t) in results.items():
    print(f"{name:34s} {100 * acc:8.1f}% {t:9.1f}")

_, acc_plain = evaluate(ft_model, ft_test_loader)
_, acc_tta = evaluate(ft_model, ft_test_loader, tta=True)
print(f"\nfine-tuned model: {100 * acc_plain:.2f}% -> {100 * acc_tta:.2f}% with horizontal-flip TTA")

Reading the table: the linear probe is the fastest and, with pretrained features, already respectable;
full fine-tuning is the most accurate; gradual unfreezing lands in between on this easy target (it shines
when the target data is scarce or noisy, where full fine-tuning overfits); training from scratch is not
competitive at this budget. Which strategy to pick is the decision matrix of Section 1.2 in numbers.

Finally, we look at the 512-dimensional penultimate features with **t-SNE** (van der Maaten and Hinton,
2008), a non-linear projection to two dimensions that tries to keep neighbours close. We compare the
frozen ImageNet features from Section 3.2 with the features of the fine-tuned model on the same test
images.

In [ ]:
def penultimate_features(model, loader):
    """Features before the head of a torchvision ResNet, by temporarily replacing fc with the identity."""
    head, model.fc = model.fc, nn.Identity()
    feats, labels = extract_features(model, loader)
    model.fc = head
    return feats, labels


ft_feats, ft_labels = penultimate_features(ft_model, ft_test_loader)
n_tsne = min(1_000, len(test_feats))
colours = [plt.cm.tab10(int(l)) for l in test_labels[:n_tsne]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
for ax, (name, feats) in zip(axes, [("ImageNet features (frozen)", test_feats), ("fine-tuned features", ft_feats)]):
    emb = TSNE(n_components=2, init="pca", perplexity=30, random_state=SEED).fit_transform(to_np(feats[:n_tsne]))
    ax.scatter(emb[:, 0], emb[:, 1], c=colours, s=8)
    ax.set_title(f"t-SNE of penultimate features: {name}"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
handles = [plt.Line2D([], [], marker="o", linestyle="", color=plt.cm.tab10(i), label=CLASSES[i]) for i in range(10)]
fig.legend(handles=handles, loc="lower center", ncol=10, fontsize=8)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

The frozen ImageNet features already group the classes into loose clouds, with the animal classes
overlapping (ImageNet distinguishes 120 dog breeds, so "dog versus cat" is not what its features are
sharpest at). After fine-tuning the clusters are tight and well separated, with the residual confusion
between cats and dogs, and between automobiles and trucks, visible as neighbouring or slightly
interleaved clusters. t-SNE is a visualisation, not a measurement (distances between clusters are not
meaningful and the layout changes with the random seed), but it is a quick way to see whether a
representation is doing its job.

> **Common pitfall.** Fine-tuning with the learning rate you would use from scratch. With SGD at 0.05 or
> 0.1 the pretrained ResNet loses several points and sometimes never recovers within a short budget;
> $10^{-2}$ for the head and lower for the backbone is a good default, and $10^{-3}$–$10^{-4}$ with AdamW.

## 4. Explaining predictions with Grad-CAM

### 4.1 From occlusion maps to gradients

Notebook 06 · Convolutional Neural Networks explained a prediction by occluding patches of the image and watching the probability drop,
which needs one forward pass per position. **Grad-CAM** (Gradient-weighted Class Activation Mapping,
Selvaraju et al., 2017) produces a comparable heatmap from a *single* forward and backward pass by using
the gradients of the class score with respect to the last convolutional feature maps.

Let $A^k \in \mathbb{R}^{u \times v}$ be the $k$-th feature map of the chosen layer (for ResNet-18's
`layer4` at $224 \times 224$ input, $K = 512$ maps of $7 \times 7$) and $y^c$ the logit of class $c$.
Grad-CAM computes an importance weight per map by global-average-pooling the gradient,

$$
\alpha_k^c = \frac{1}{uv} \sum_{i=1}^{u} \sum_{j=1}^{v} \frac{\partial y^c}{\partial A^k_{ij}},
$$

and then a weighted sum of the maps, passed through a ReLU:

$$
L^c_{\text{Grad-CAM}} = \mathrm{ReLU}\Big(\sum_k \alpha_k^c A^k\Big) \in \mathbb{R}^{u \times v} .
$$

The intuition: $\alpha_k^c$ measures how much increasing map $k$ everywhere would increase the class
score, so maps with large positive $\alpha$ are the ones that argue *for* class $c$; the ReLU discards
regions whose net effect is negative (they argue for other classes). The $u \times v$ map is bilinearly
upsampled to the image size and overlaid as a heatmap.

Why the last convolutional layer? It is the deepest layer that still has a spatial layout, so it offers
the best compromise between semantics (its channels respond to object parts) and localisation (its
$7 \times 7$ grid). Earlier layers give sharper but less class-specific maps. For architectures that end
in global average pooling followed by one linear layer (ResNets included), Grad-CAM at the last layer
coincides exactly with the older CAM method of Zhou et al. (2016): the gradient of $y^c$ with respect to the
pooled feature $k$ is simply the classifier weight $w_{ck}$, so $\alpha_k^c = w_{ck} / (uv)$.

### 4.2 Implementation with hooks

We need two things during one forward/backward pass: the activations $A$ of the target layer and the
gradient $\partial y^c / \partial A$. A **forward hook** on the layer captures its output; on that output
tensor we register a **tensor hook**, which autograd calls during the backward pass with the gradient
flowing into it. The class below wraps this and removes the hook when we are done, because a forgotten
hook keeps firing on every later forward pass.

In [ ]:
class GradCAM:
    """Grad-CAM for a given layer of a model: one forward hook plus a tensor gradient hook."""

    def __init__(self, model, target_layer):
        self.model, self.activations, self.gradients = model, None, None
        self.handle = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inputs, output):
        self.activations = output.detach()
        if output.requires_grad:                               # only when autograd is on
            output.register_hook(self._grad_hook)

    def _grad_hook(self, grad):
        self.gradients = grad.detach()

    def __call__(self, x, class_idx=None):
        """Returns (cam maps (N, H, W) in [0, 1], logits, class indices used)."""
        self.model.eval()
        x = x.clone().requires_grad_(True)
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1)
        class_idx = torch.as_tensor(class_idx, device=logits.device).view(-1)
        score = logits.gather(1, class_idx[:, None]).sum()      # sum of the chosen logits
        self.model.zero_grad(set_to_none=True)
        score.backward()
        weights = self.channel_weights()                                      # alpha_k^c: (N, K, 1, 1)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))   # (N, 1, u, v)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)
        return cam.detach(), logits.detach(), class_idx

    def channel_weights(self):
        """Grad-CAM weights: the global average of the gradients (overridden by Grad-CAM++ in Exercise 2)."""
        return self.gradients.mean(dim=(2, 3), keepdim=True)

    def remove(self):
        self.handle.remove()


display_tf = v2.Compose([v2.Resize(256, antialias=True), v2.CenterCrop(224)])    # same geometry as `preprocess`

cam_engine = GradCAM(pretrained, pretrained.layer4)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for row, (name, img) in enumerate(sample_images.items()):
    x = preprocess(to_chw_uint8(img))[None].to(device)
    cam, logits, cls = cam_engine(x)
    top2 = logits[0].topk(2).indices
    cam2, _, _ = cam_engine(x, class_idx=top2[1])                            # map for the runner-up class
    shown = to_np(display_tf(to_chw_uint8(img)).permute(1, 2, 0))
    axes[row, 0].imshow(shown); axes[row, 0].set_title(name)
    axes[row, 1].imshow(shown); axes[row, 1].imshow(to_np(cam[0]), cmap="jet", alpha=0.5)
    axes[row, 1].set_title(f"Grad-CAM: {categories[cls[0].item()][:22]}")
    axes[row, 2].imshow(shown); axes[row, 2].imshow(to_np(cam2[0]), cmap="jet", alpha=0.5)
    axes[row, 2].set_title(f"Grad-CAM: {categories[top2[1].item()][:22]} (2nd)")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()
cam_engine.remove()
print("activations:", tuple(cam_engine.activations.shape), "| gradients:", tuple(cam_engine.gradients.shape))

The map for the top class covers the building (or the flower), the evidence for the predicted class, and
the map for the second class typically highlights a different region or a different part of the same
object. Grad-CAM is **class-discriminative**: it explains *why this class* rather than showing what the
network looks at in general, and asking for two classes on the same image is the standard way to
demonstrate it.

Let us also apply Grad-CAM to our fine-tuned CIFAR model. At $96 \times 96$ input, `layer4` is only
$3 \times 3$, so the maps are coarse; this is a real limitation of the method at low resolution.

In [ ]:
cam_engine = GradCAM(ft_model, ft_model.layer4)
xs = torch.stack([ft_eval_tf(cifar_test_x[i]) for i in range(6)]).to(device)
cams, logits, cls = cam_engine(xs)
cam_engine.remove()

fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
for i in range(6):
    raw = to_np(cifar_test_x[i].permute(1, 2, 0))
    big = to_np(v2.functional.resize(cifar_test_x[i], [FT_SIZE, FT_SIZE], antialias=True).permute(1, 2, 0))
    axes[0, i].imshow(raw); axes[0, i].set_title(f"true: {CLASSES[cifar_test_y[i]]}", fontsize=9)
    axes[1, i].imshow(big); axes[1, i].imshow(to_np(cams[i]), cmap="jet", alpha=0.5)
    axes[1, i].set_title(f"pred: {CLASSES[cls[i].item()]}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

### 4.3 What Grad-CAM can and cannot tell you

Grad-CAM answers "where in the image is the evidence for class $c$?" at the resolution of the chosen
layer. That is useful for catching shortcuts: a model that classifies "ship" by looking at water, or a
medical model that reads the hospital's watermark instead of the scan, shows it immediately. But there are
important caveats:

- **Resolution.** The map is $7 \times 7$ at 224 px and $3 \times 3$ at 96 px; small objects and fine
  boundaries are lost. Guided Grad-CAM combines it with a pixel-level gradient method for sharper maps,
  and Grad-CAM++ (Exercise 2) refines the weights for images with several instances of the class.
- **Only positive evidence.** The ReLU discards regions that argue against the class, and the global
  average of the gradient can cancel positive and negative contributions within one map.
- **Faithfulness is not guaranteed.** A heatmap that looks plausible is not proof that the model uses that
  region, and Adebayo et al. (2018) showed that some popular saliency methods barely change when the
  model's weights are randomised. Grad-CAM passes those sanity checks better than most, but any
  explanation should be validated, for instance by occluding the highlighted region and checking that the
  prediction actually changes.

Notebook 16 · Interpretability, Robustness and Uncertainty treats attribution methods systematically,
including the sanity checks, integrated gradients and their evaluation.

## 5. Vision Transformers

### 5.1 An image is a sequence of patches

The Transformer (Vaswani et al., 2017) was designed for sequences of word tokens, and
Notebook 09 · Attention and Transformers derives it in full. Dosovitskiy et al. (2021) showed that it
works on images with almost no modification, provided the image is first turned into a sequence: the
**Vision Transformer** (ViT). The recipe is short.

1. **Patchify.** Cut the image $x \in \mathbb{R}^{C \times H \times W}$ into non-overlapping patches of
   size $P \times P$. There are $N = HW / P^2$ patches; each is flattened into a vector of length
   $P^2 C$. For a $32 \times 32$ CIFAR image with $P = 4$ this gives $N = 64$ patches of 48 values;
   for a $224 \times 224$ ImageNet image with $P = 16$, $196$ patches of 768 values.
2. **Embed.** Each flattened patch is mapped to a $D$-dimensional token by a linear map $E$. A linear map
   applied to every non-overlapping $P \times P$ patch is exactly a convolution with kernel size $P$ and
   stride $P$, so the patch embedding is one `nn.Conv2d(C, D, kernel_size=P, stride=P)`, followed by a
   reshape to $(N, D)$.
3. **Class token and positions.** A learned vector $x_{\text{cls}} \in \mathbb{R}^D$ is prepended to the
   sequence; its final state will be the image representation. Because self-attention is
   permutation-equivariant (shuffling the input tokens merely shuffles the outputs in the same way, so
   the network cannot tell where a patch came from), a learned **positional embedding**
   $E_{\text{pos}} \in \mathbb{R}^{(N+1) \times D}$ is added to the tokens:
   $z_0 = [x_{\text{cls}};\, x_p^1 E;\, \dots;\, x_p^N E] + E_{\text{pos}}$.
4. **Encoder.** $L$ identical Transformer blocks update the tokens. Each block applies **multi-head
   self-attention** (every token gathers information from every other token, weighted by learned
   similarity) and then a position-wise **MLP**, both with residual connections and LayerNorm in front
   (pre-norm):
   $z' = z + \mathrm{MSA}(\mathrm{LN}(z)), \qquad z_{\text{out}} = z' + \mathrm{MLP}(\mathrm{LN}(z')).$
5. **Classify.** A LayerNorm and a linear layer on the final class token give the logits.

Self-attention for one head computes queries, keys and values by linear maps of the tokens,
$Q = ZW_Q,\ K = ZW_K,\ V = ZW_V$, and outputs

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V ,
$$

a weighted average of the values in which token $i$'s weights are the softmax of its query's dot products
with all keys. With $h$ heads the $D$ channels are split into $h$ groups of $d = D/h$, each with its own
attention pattern, and the results are concatenated and mixed by one more linear map. The cost of the
$QK^\top$ product is $O(N^2 D)$: quadratic in the number of tokens, which is why the patch size matters
(Exercise 3).

### 5.2 A compact implementation

The code below is a complete ViT in about sixty lines. `MultiHeadSelfAttention` keeps the last attention
weights so that we can visualise them; everything else is standard. Notebook 09 · Attention and Transformers
revisits every component (masking, positional encodings, the choice of pre- versus post-norm) in depth.

In [ ]:
class PatchEmbedding(nn.Module):
    """(N, C, H, W) -> (N, num_patches, dim) via a conv with kernel = stride = patch size."""

    def __init__(self, in_ch=3, patch=4, dim=128):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, dim, kernel_size=patch, stride=patch)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)        # (N, dim, h, w) -> (N, h*w, dim)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        assert dim % heads == 0
        self.heads, self.scale = heads, (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.attn_weights = None                              # (N, heads, L, L) of the last forward pass

    def forward(self, x):
        N, L, D = x.shape
        qkv = self.qkv(x).view(N, L, 3, self.heads, D // self.heads).permute(2, 0, 3, 1, 4)   # (3, N, h, L, d)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale        # (N, h, L, L)
        attn = attn.softmax(dim=-1)
        self.attn_weights = attn.detach()
        out = (attn @ v).transpose(1, 2).reshape(N, L, D)     # concatenate the heads
        return self.proj(out)


class TransformerBlock(nn.Module):
    """Pre-norm Transformer encoder block: x + MSA(LN(x)), then x + MLP(LN(x))."""

    def __init__(self, dim, heads, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(int(dim * mlp_ratio), dim), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class ViT(nn.Module):
    """A small Vision Transformer for square images."""

    def __init__(self, image_size=32, patch=4, dim=128, depth=4, heads=4, num_classes=10,
                 mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(3, patch, dim)
        n_patches = (image_size // patch) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(dim, heads, mlp_ratio, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        tokens = self.patch_embed(x)                                     # (N, num_patches, dim)
        cls = self.cls_token.expand(x.shape[0], -1, -1)                  # (N, 1, dim)
        z = self.dropout(torch.cat([cls, tokens], dim=1) + self.pos_embed)
        for block in self.blocks:
            z = block(z)
        return self.head(self.norm(z[:, 0]))                             # class token -> logits


VIT_DIM, VIT_DEPTH, VIT_HEADS = (96, 3, 4) if FAST_RUN else (192, 6, 6)
torch.manual_seed(SEED)
vit = ViT(image_size=32, patch=4, dim=VIT_DIM, depth=VIT_DEPTH, heads=VIT_HEADS)
x = torch.zeros(2, 3, 32, 32)
with torch.no_grad():
    tokens = vit.patch_embed(x)
    print(f"image {tuple(x.shape)} -> patch tokens {tuple(tokens.shape)} -> with class token "
          f"{tuple(torch.cat([vit.cls_token.expand(2, -1, -1), tokens], 1).shape)} -> logits {tuple(vit(x).shape)}")
print(f"ViT: dim {VIT_DIM}, depth {VIT_DEPTH}, heads {VIT_HEADS}, {count_params(vit):,} parameters")

### 5.3 ViT versus CNN on CIFAR-10

We train the ViT from scratch on CIFAR-10 at $32 \times 32$ with the augmentation of
Notebook 06 · Convolutional Neural Networks and
AdamW (learning rate $10^{-3}$, weight decay 0.05, 10 % linear warm-up then cosine decay: Transformers
are sensitive to the early learning rate and warm-up is essential). As a baseline we train a small
VGG-style CNN with the *same* optimiser, schedule, data and epochs. Under `FAST_RUN` both see 4,000
images once; the full run uses the whole training set for 12 epochs.

In [ ]:
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)     # computed in Notebook 06
c32_train_tf = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip(),
                           v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)])
c32_eval_tf = v2.Compose([v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)])

N_VIT_TRAIN = 4_000 if FAST_RUN else 50_000
N_VIT_TEST = 1_000 if FAST_RUN else 10_000
EPOCHS_VIT = 1 if FAST_RUN else 12
vit_train_loader, vit_test_loader = make_cifar_loaders(N_VIT_TRAIN, N_VIT_TEST, c32_train_tf, c32_eval_tf, batch_size=128)


def conv_bn_relu(in_ch, out_ch):
    return nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))


class SmallCNN(nn.Module):
    """Three VGG-style stages (two conv-BN-ReLU each + max pooling), global average pooling, linear."""

    def __init__(self, widths=(32, 64, 128), num_classes=10):
        super().__init__()
        layers, c = [], 3
        for w in widths:
            layers += [conv_bn_relu(c, w), conv_bn_relu(w, w), nn.MaxPool2d(2)]
            c = w
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, num_classes))

    def forward(self, x):
        return self.head(self.features(x))


CNN_W = 16 if FAST_RUN else 32
compare = {}
for name, make in [("ViT", lambda: ViT(32, 4, VIT_DIM, VIT_DEPTH, VIT_HEADS)),
                   ("CNN", lambda: SmallCNN((CNN_W, 2 * CNN_W, 4 * CNN_W)))]:
    torch.manual_seed(SEED)
    model = make()
    print(f"--- {name}: {count_params(model):,} parameters, {EPOCHS_VIT} epoch(s) on {N_VIT_TRAIN:,} images")
    hist = fit(model, vit_train_loader, vit_test_loader, EPOCHS_VIT, lr=1e-3, weight_decay=0.05,
               optimizer="adamw", warmup_frac=0.1)
    compare[name] = (model, hist)
vit = compare["ViT"][0]

print(f"\n{'model':6s} {'params':>10s} {'s/epoch':>8s} {'test acc':>9s}")
for name, (model, hist) in compare.items():
    print(f"{name:6s} {count_params(model):>10,} {np.mean(hist['epoch_time']):8.1f} {100 * hist['test_acc'][-1]:8.1f}%")

In the full run the CNN wins clearly (roughly 85-88 % against 70-78 % for the ViT after 12 epochs),
even though the ViT has more parameters and costs more time per epoch. This is the expected result and
the most important thing to understand about Vision Transformers: **they have almost no inductive bias
about images**. A CNN "knows" that nearby pixels are related and that a feature is the same feature
wherever it appears (Notebook 06 · Convolutional Neural Networks, Section 1); a ViT must learn locality and translation equivariance
from data, which it can, but only given a lot of it. In the original paper, ViT-B/16 trained on ImageNet-1k
alone (1.3 M images) was *worse* than a comparable ResNet, roughly equal on ImageNet-21k (14 M) and better
on JFT-300M (300 M). On 50,000 CIFAR images the CNN's priors are worth far more than the ViT's flexibility.

Three developments made ViTs practical nonetheless. **DeiT** (Touvron et al., 2021) showed that strong
augmentation, regularisation and distillation from a CNN teacher let a ViT match CNNs on ImageNet-1k.
**Pretraining** at scale, supervised or self-supervised (Section 7), then fine-tuning, is now the default,
and pretrained ViTs transfer at least as well as CNNs. And hybrid designs (Swin Transformers with local
attention windows, ConvNeXt going the other way) show that the two families are converging on similar
design points. The take-away for practice is simple: train a CNN from scratch on a small dataset; use a
pretrained model, CNN or ViT, whenever one exists for your domain.

Let us look at what the ViT attends to. The class token's attention weights over the 64 patches, averaged
over the heads of the last block, form an $8 \times 8$ map that we overlay on the image.

In [ ]:
vit.eval()
xs = torch.stack([c32_eval_tf(cifar_test_x[i]) for i in range(6)]).to(device)
with torch.no_grad():
    logits = vit(xs).float()
attn = vit.blocks[-1].attn.attn_weights.float()                    # (N, heads, 65, 65)
cls_attn = attn[:, :, 0, 1:].mean(dim=1).view(-1, 1, 8, 8)         # class token -> patches, mean over heads
cls_attn = F.interpolate(cls_attn, size=(32, 32), mode="bilinear", align_corners=False)[:, 0]
cls_attn = cls_attn / cls_attn.amax(dim=(1, 2), keepdim=True)

fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
for i in range(6):
    raw = to_np(cifar_test_x[i].permute(1, 2, 0))
    axes[0, i].imshow(raw); axes[0, i].set_title(f"true: {CLASSES[cifar_test_y[i]]}", fontsize=9)
    axes[1, i].imshow(raw); axes[1, i].imshow(to_np(cls_attn[i]), cmap="jet", alpha=0.5)
    axes[1, i].set_title(f"pred: {CLASSES[logits[i].argmax().item()]}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Class-token attention of the last block (mean over heads)"); plt.tight_layout(); plt.show()

After a full training run the class token attends mostly to patches on the object, a rough
segmentation that emerges without any localisation supervision; under `FAST_RUN` the maps are still
close to uniform. Self-supervised ViTs (DINO, Section 7) produce strikingly clean versions of these maps,
which is one reason ViTs have become the backbone of choice for dense tasks as well.

> **Going deeper.** Two more things about ViTs are worth knowing. The positional embedding is learned for
> one image size; to fine-tune at a higher resolution it is bilinearly interpolated to the new number of
> patches. And the quadratic attention cost means that 224-px images with $P = 16$ (196 tokens) are cheap
> but $P = 8$ (784 tokens) is 16x more expensive in the attention layers; efficient-attention variants are
> discussed in Notebook 09 · Attention and Transformers.

## 6. Beyond classification: segmentation and detection

### 6.1 Semantic segmentation and the U-Net

**Semantic segmentation** assigns a class to *every pixel*: the output for an image of size $H \times W$
is a map of shape $(K, H, W)$ of class logits, and the loss is the per-pixel cross-entropy averaged over
all pixels. A classifier's backbone is not directly usable, because it throws away spatial resolution
on purpose: ResNet-18 reduces $224 \times 224$ to $7 \times 7$. Segmentation networks need both the
*context* that deep, low-resolution features provide (what object is this?) and the *localisation* that
shallow, high-resolution features provide (where exactly is its boundary?).

The **U-Net** (Ronneberger, Fischer and Brox, 2015) combines the two with an encoder-decoder shape and
**skip connections**: the encoder is a normal CNN that halves the resolution at each level; the decoder
upsamples step by step with transposed convolutions (Notebook 06 · Convolutional Neural Networks, Section 3); and at each level the
decoder's feature map is *concatenated* with the encoder's feature map of the same resolution before the
next convolutions. The concatenation gives the decoder direct access to fine detail, and the gradient a
short path to the early layers. Drawn with the encoder going down on the left and the decoder coming up on
the right, the architecture looks like a U, hence the name. It was designed for biomedical images with
very few training examples, and it remains the default segmentation architecture (and, with time
conditioning, the backbone of diffusion models in Notebook 12 · GANs and Diffusion Models).

Two metrics are standard for segmentation, both computed per class on the binary masks $A$ (prediction)
and $B$ (ground truth):

$$
\mathrm{IoU} = \frac{|A \cap B|}{|A \cup B|}, \qquad
\mathrm{Dice} = \frac{2\,|A \cap B|}{|A| + |B|}, \qquad
\mathrm{Dice} = \frac{2\,\mathrm{IoU}}{1 + \mathrm{IoU}} .
$$

Both are 1 for a perfect mask and 0 for a disjoint one; Dice is always at least IoU and penalises small
errors less. Pixel accuracy is *not* a good metric, because the background usually dominates: a network
that predicts "background" everywhere can score 90 % pixel accuracy with an IoU of 0 on every object.

### 6.2 A synthetic shapes dataset

To train a U-Net in a minute we generate our own data: $64 \times 64$ images with a random background
colour, one to three randomly coloured circles and rectangles, and Gaussian noise. The mask has three
classes: 0 background, 1 circle, 2 rectangle. Because colours are random, the network cannot use colour
to tell circles from rectangles; it has to learn *shape*, which is a genuinely spatial task.

In [ ]:
def make_shapes(n, size=64, seed=0):
    """n images (n, 3, size, size) in [0, 1] and masks (n, size, size) with classes 0/1/2."""
    rng = np.random.default_rng(seed)
    images = np.zeros((n, 3, size, size), dtype=np.float32)
    masks = np.zeros((n, size, size), dtype=np.int64)
    yy, xx = np.mgrid[0:size, 0:size]
    for i in range(n):
        img = np.ones((3, size, size), dtype=np.float32) * rng.uniform(0.1, 0.9, size=(3, 1, 1)).astype(np.float32)
        for _ in range(int(rng.integers(1, 4))):
            kind = int(rng.integers(0, 2))                        # 0: circle, 1: rectangle
            colour = rng.uniform(0.0, 1.0, size=3).astype(np.float32)
            if kind == 0:
                cx, cy, r = rng.integers(10, size - 10), rng.integers(10, size - 10), rng.integers(5, 12)
                region = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
            else:
                w, h = rng.integers(8, 24, size=2)
                x0, y0 = rng.integers(0, size - w), rng.integers(0, size - h)
                region = np.zeros((size, size), dtype=bool)
                region[y0:y0 + h, x0:x0 + w] = True
            img[:, region] = colour[:, None]
            masks[i][region] = kind + 1
        img += rng.normal(0.0, 0.06, size=img.shape).astype(np.float32)
        images[i] = np.clip(img, 0.0, 1.0)
    return torch.from_numpy(images), torch.from_numpy(masks)


N_SEG_TRAIN = 256 if FAST_RUN else 2_000
N_SEG_TEST = 64 if FAST_RUN else 200
EPOCHS_SEG = 2 if FAST_RUN else 8
seg_train_x, seg_train_y = make_shapes(N_SEG_TRAIN, seed=SEED)
seg_test_x, seg_test_y = make_shapes(N_SEG_TEST, seed=SEED + 1)
seg_train_loader = DataLoader(TensorDataset(seg_train_x, seg_train_y), batch_size=16, shuffle=True, num_workers=0, pin_memory=PIN)
seg_test_loader = DataLoader(TensorDataset(seg_test_x, seg_test_y), batch_size=32, shuffle=False, num_workers=0, pin_memory=PIN)
print(f"images {tuple(seg_train_x.shape)} {seg_train_x.dtype}, masks {tuple(seg_train_y.shape)} {seg_train_y.dtype}; "
      f"class pixel shares: {[f'{v:.2f}' for v in torch.bincount(seg_train_y.flatten(), minlength=3).float().div(seg_train_y.numel()).tolist()]}")

fig, axes = plt.subplots(2, 6, figsize=(13, 4.4))
for i in range(6):
    axes[0, i].imshow(to_np(seg_train_x[i].permute(1, 2, 0))); axes[0, i].set_title(f"image {i}", fontsize=9)
    axes[1, i].imshow(to_np(seg_train_y[i]), cmap="viridis", vmin=0, vmax=2); axes[1, i].set_title("mask (0/1/2)", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

Close to 90 % of the pixels are background (the printed shares say exactly how many), which is why we will
judge the model by Dice and IoU rather than pixel accuracy. Now the U-Net: three encoder levels (64, 32, 16 pixels) with 16, 32 and 64
channels, a bottleneck at 8 pixels with 128 channels, and a mirrored decoder whose convolutions see the
concatenation of the upsampled map and the skip connection (hence their doubled input channels).

In [ ]:
class DoubleConv(nn.Module):
    """(conv3x3 -> BN -> ReLU) x 2, the basic unit of the U-Net."""

    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(conv_bn_relu(in_ch, out_ch), conv_bn_relu(out_ch, out_ch))

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """A small U-Net: 3 encoder levels + bottleneck, transposed-conv upsampling, concatenation skips."""

    def __init__(self, in_ch=3, num_classes=3, base=16):
        super().__init__()
        b = base
        self.enc1, self.enc2, self.enc3 = DoubleConv(in_ch, b), DoubleConv(b, 2 * b), DoubleConv(2 * b, 4 * b)
        self.bottleneck = DoubleConv(4 * b, 8 * b)
        self.pool = nn.MaxPool2d(2)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * b, 4 * b, 2, stride=2), DoubleConv(8 * b, 4 * b)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, stride=2), DoubleConv(4 * b, 2 * b)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * b, b, 2, stride=2), DoubleConv(2 * b, b)
        self.out = nn.Conv2d(b, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)                                  # (b,   64, 64)
        e2 = self.enc2(self.pool(e1))                      # (2b,  32, 32)
        e3 = self.enc3(self.pool(e2))                      # (4b,  16, 16)
        z = self.bottleneck(self.pool(e3))                 # (8b,   8,  8)
        d3 = self.dec3(torch.cat([self.up3(z), e3], dim=1))    # (4b, 16, 16)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))   # (2b, 32, 32)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))   # (b,  64, 64)
        return self.out(d1)                                # (num_classes, 64, 64) logits per pixel


torch.manual_seed(SEED)
unet = UNet()
with torch.no_grad():
    print("U-Net:", tuple(seg_train_x[:2].shape), "->", tuple(unet(seg_train_x[:2]).shape), f"| {count_params(unet):,} parameters")

### 6.3 Training with Dice and IoU

The per-pixel cross-entropy is `F.cross_entropy` applied to logits of shape $(N, K, H, W)$ and targets of
shape $(N, H, W)$; PyTorch handles the extra spatial dimensions natively. The metric function computes,
for each image and class, intersection and union of the predicted and true masks, and reports the mean
Dice and IoU over the three classes (a class absent from both masks counts as a perfect 1, the usual
convention; Exercise 4 implements the dataset-level alternative). We train with AdamW and the same
warm-up/cosine schedule as before.

In [ ]:
@torch.no_grad()
def segmentation_scores(logits, target, num_classes=3, eps=1e-6):
    """Per-image mean Dice and IoU over classes. Returns two tensors of shape (N,)."""
    pred = logits.argmax(dim=1)
    dices, ious = [], []
    for c in range(num_classes):
        p, t = pred == c, target == c
        inter = (p & t).sum(dim=(1, 2)).float()
        union = (p | t).sum(dim=(1, 2)).float()
        size = p.sum(dim=(1, 2)).float() + t.sum(dim=(1, 2)).float()
        dices.append((2 * inter + eps) / (size + eps))
        ious.append((inter + eps) / (union + eps))
    return torch.stack(dices, dim=1).mean(dim=1), torch.stack(ious, dim=1).mean(dim=1)


@torch.no_grad()
def evaluate_segmentation(model, loader):
    model.eval()
    loss_sum, dice_sum, iou_sum, n = 0.0, 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(x).float()
        loss_sum += F.cross_entropy(logits, y).item() * x.size(0)
        dice, iou = segmentation_scores(logits, y)
        dice_sum += dice.sum().item(); iou_sum += iou.sum().item(); n += x.size(0)
    return loss_sum / n, dice_sum / n, iou_sum / n


def fit_segmentation(model, train_loader, test_loader, epochs, lr=2e-3):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    total, warm = epochs * len(train_loader), max(1, int(0.05 * epochs * len(train_loader)))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    hist = {"train_loss": [], "test_loss": [], "dice": [], "iou": []}
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); loss_sum, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
                loss = F.cross_entropy(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            loss_sum += loss.item() * x.size(0); n += x.size(0)
        te_loss, dice, iou = evaluate_segmentation(model, test_loader)
        hist["train_loss"].append(loss_sum / n); hist["test_loss"].append(te_loss); hist["dice"].append(dice); hist["iou"].append(iou)
        print(f"epoch {epoch:2d}/{epochs} | train loss {loss_sum / n:.4f} | test loss {te_loss:.4f} "
              f"dice {dice:.3f} IoU {iou:.3f} | {time.time() - t0:5.1f}s")
    return hist


torch.manual_seed(SEED)
unet = UNet()
hist_unet = fit_segmentation(unet, seg_train_loader, seg_test_loader, EPOCHS_SEG)

The Dice and IoU columns should climb well above what "predict background everywhere" would give. That
trivial model has a pixel accuracy of about 0.89 (the background share), but under our per-image
convention its mean IoU is only about 0.5: the background class scores about 0.89, and each shape class
scores 0 on every image that contains it and the free 1 only on the roughly 30 % of images where it is
absent, so $(0.89 + 0.3 + 0.3)/3 \approx 0.5$. Under the dataset-level convention of Exercise 4, where
an absent class earns nothing, the same trivial model gets $0.89 / 3 \approx 0.3$. Let us look at the
predicted masks for six test images next to the ground truth, with the per-image IoU in the title.

In [ ]:
unet.eval()
with torch.no_grad():
    seg_logits = unet(seg_test_x[:6].to(device)).float().cpu()
preds = seg_logits.argmax(dim=1)
dice, iou = segmentation_scores(seg_logits, seg_test_y[:6])
fig, axes = plt.subplots(3, 6, figsize=(13, 6.6))
for i in range(6):
    axes[0, i].imshow(to_np(seg_test_x[i].permute(1, 2, 0))); axes[0, i].set_title(f"test image {i}", fontsize=9)
    axes[1, i].imshow(to_np(seg_test_y[i]), cmap="viridis", vmin=0, vmax=2); axes[1, i].set_title("ground truth", fontsize=9)
    axes[2, i].imshow(to_np(preds[i]), cmap="viridis", vmin=0, vmax=2)
    axes[2, i].set_title(f"prediction (IoU {iou[i]:.2f})", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

After the full run the U-Net segments circles and rectangles almost perfectly (mean IoU above 0.9), with
the remaining errors at object boundaries and where shapes overlap. Even two epochs on 256 images under
`FAST_RUN` produce recognisable masks, because the skip connections make boundary information available
to the decoder from the first iteration. Real segmentation problems add three difficulties that this toy
task lacks: class imbalance severe enough to need a Dice or focal loss instead of plain cross-entropy,
much larger images that must be processed in tiles, and a pretrained encoder (a ResNet, or a ViT) in
place of our small one, which is transfer learning again.

### 6.4 Object detection: concepts

**Object detection** outputs a set of bounding boxes with class labels and confidence scores. It is
harder than classification and segmentation because the output is a *set* of variable size, which does
not fit the fixed-shape tensor that a network naturally produces. Every detector is a way around this
problem, and the family tree has three main branches.

**Two-stage detectors** (R-CNN → Fast R-CNN → **Faster R-CNN**, Ren et al., 2015) first propose regions
that might contain objects, then classify each region and refine its box. Faster R-CNN's *region
proposal network* slides over the backbone's feature map and, at each position, scores a set of
**anchors**: reference boxes of several sizes and aspect ratios. Proposals with high objectness are cut
out of the feature map (RoI pooling / RoI align), classified and regressed to a better box. Two-stage
detectors are accurate but slow.

**One-stage detectors** (**YOLO**, Redmon et al., 2016; SSD; RetinaNet with the focal loss, which fixes
the extreme foreground/background imbalance of dense prediction) skip the proposal stage and predict
boxes and classes directly at every anchor of a dense grid, in a single pass. They are fast enough for
video, and modern versions (YOLOv5-v8, anchor-free FCOS) are competitive in accuracy.

**Set prediction with Transformers** (**DETR**, Carion et al., 2020) removes anchors entirely: a
Transformer decoder with $M$ learned "object queries" outputs $M$ (box, class) pairs, and training uses
the Hungarian algorithm to match predictions to ground-truth objects one-to-one, with "no object" for the
rest. DETR needs no hand-designed anchors and no post-processing, at the price of slow training.

Anchor-based detectors produce many overlapping boxes for the same object (each anchor near it fires), so
they all end with **non-maximum suppression** (NMS): sort the boxes by score; repeatedly take the highest
scoring box, keep it, and discard every remaining box that overlaps it with IoU above a threshold
(typically 0.5); stop when no box is left. The box overlap is the same IoU as in segmentation, applied to
rectangles: for boxes given as $(x_1, y_1, x_2, y_2)$ the intersection is the rectangle between the
element-wise maximum of the top-left corners and the element-wise minimum of the bottom-right corners,
clamped at zero width and height.

In [ ]:
def box_iou(a, b):
    """IoU between boxes a (N, 4) and b (M, 4) in (x1, y1, x2, y2) format -> (N, M)."""
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    top_left = torch.max(a[:, None, :2], b[None, :, :2])          # (N, M, 2)
    bottom_right = torch.min(a[:, None, 2:], b[None, :, 2:])      # (N, M, 2)
    wh = (bottom_right - top_left).clamp(min=0)
    inter = wh[..., 0] * wh[..., 1]
    return inter / (area_a[:, None] + area_b[None, :] - inter)


def nms(boxes, scores, iou_threshold=0.5):
    """Greedy non-maximum suppression; returns the indices of the kept boxes, best first."""
    order = scores.argsort(descending=True)
    keep = []
    while order.numel() > 0:
        i = order[0]
        keep.append(i.item())
        if order.numel() == 1:
            break
        ious = box_iou(boxes[i][None], boxes[order[1:]])[0]
        order = order[1:][ious <= iou_threshold]                  # drop everything that overlaps too much
    return torch.tensor(keep, dtype=torch.long)


# synthetic proposals: three objects, six jittered boxes each, random scores
torch.manual_seed(SEED)
true_boxes = torch.tensor([[10., 10., 40., 45.], [55., 20., 90., 50.], [30., 60., 80., 95.]])
boxes, scores = [], []
for tb in true_boxes:
    for _ in range(6):
        boxes.append(tb + torch.randn(4) * 3); scores.append(torch.rand(1).item() * 0.6 + 0.4)
boxes, scores = torch.stack(boxes), torch.tensor(scores)

keep = nms(boxes, scores, 0.5)
keep_ref = torchvision.ops.nms(boxes, scores, 0.5)
print("IoU check   : max |ours - torchvision.ops.box_iou| =", f"{(box_iou(boxes, boxes) - torchvision.ops.box_iou(boxes, boxes)).abs().max().item():.1e}")
print("NMS (ours)  :", keep.tolist())
print("NMS (torch) :", keep_ref.tolist(), "| identical:", torch.equal(keep, keep_ref))

from matplotlib.patches import Rectangle
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, idx, title in [(axes[0], range(len(boxes)), f"{len(boxes)} proposals"), (axes[1], keep.tolist(), f"{len(keep)} boxes after NMS (IoU > 0.5 suppressed)")]:
    for j in idx:
        x1, y1, x2, y2 = boxes[j].tolist()
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=plt.cm.viridis(scores[j].item()), lw=1.5))
        ax.text(x1, y1 - 1, f"{scores[j]:.2f}", fontsize=7)
    ax.set(xlim=(0, 100), ylim=(100, 0), title=title, aspect="equal")
plt.tight_layout(); plt.show()

Our NMS keeps exactly the same boxes as `torchvision.ops.nms`: one per object, the highest-scoring
proposal of each cluster. Note that NMS has a failure mode built in: two *different* objects that overlap
heavily (a crowd of people) are also merged, which is what Soft-NMS and the matching loss of DETR address.

### 6.5 A pretrained detector (optional)

torchvision provides pretrained detectors trained on COCO (80 object categories). The cell below runs the
smallest one, a Faster R-CNN with a MobileNetV3 backbone, on the temple photograph. It is skipped under
`FAST_RUN` and guarded against a missing download; COCO's categories are everyday objects, so on this
particular image the detector may find little, which is itself a useful reminder that a detector only
knows the categories it was trained on. Set `FAST_RUN = False` and try your own photograph.

In [ ]:
if FAST_RUN:
    print("Detection demo skipped under FAST_RUN (it downloads ~75 MB of weights and runs a full detector).")
else:
    try:
        from torchvision.models.detection import (fasterrcnn_mobilenet_v3_large_320_fpn,
                                                  FasterRCNN_MobileNet_V3_Large_320_FPN_Weights)
        det_weights = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
        detector = fasterrcnn_mobilenet_v3_large_320_fpn(weights=det_weights).to(device).eval()
        det_categories = det_weights.meta["categories"]
        img_u8 = to_chw_uint8(sample_images["china.jpg"])
        with torch.no_grad():
            out = detector([det_weights.transforms()(img_u8).to(device)])[0]
        keep = out["scores"] > 0.3
        det_boxes = out["boxes"][keep].cpu()
        det_labels = [f"{det_categories[i]} {s:.2f}" for i, s in zip(out["labels"][keep].tolist(), out["scores"][keep].tolist())]
        print(f"{len(det_boxes)} detections with score > 0.3: {det_labels}")
        drawn = torchvision.utils.draw_bounding_boxes(img_u8, det_boxes, labels=det_labels, colors="red", width=3)
        plt.figure(figsize=(8, 5.5)); plt.imshow(to_np(drawn.permute(1, 2, 0))); plt.axis("off")
        plt.title("Faster R-CNN (MobileNetV3-FPN, COCO) on china.jpg"); plt.tight_layout(); plt.show()
    except Exception as e:
        print(f"Detection demo skipped: {type(e).__name__}: {str(e)[:120]}")

## 7. Self-supervised and contrastive learning

### 7.1 Learning representations without labels

Everything in Section 3 relied on ImageNet's 1.28 million *labelled* images. Labels are expensive; images
are not. **Self-supervised learning** (SSL) trains a backbone on unlabelled images by inventing a
**pretext task** whose supervision comes from the data itself, and then transfers the backbone exactly as
in Section 3. Since about 2020, self-supervised pretraining matches or beats supervised ImageNet
pretraining on most downstream tasks, and it is how the largest vision models are trained.

The dominant family of pretext tasks is **contrastive learning**: two random augmentations of the same
image (a *positive pair*) should map to nearby points in representation space, while augmentations of
*different* images (*negatives*) should map far apart. The augmentations (random resized crop, flip,
strong colour jitter, grayscale, blur) define what the representation should be invariant to; the
negatives prevent the trivial solution of mapping everything to the same point.

- **SimCLR** (Chen et al., 2020) is the cleanest version. A batch of $N$ images gives $2N$ views; an
  encoder $f$ (a ResNet) and a small projection MLP $g$ map each view to $z_i$, and the loss below pulls
  each positive pair together against the other $2N - 2$ views. It needs very large batches (4,096-8,192)
  for enough negatives; the projection head is thrown away after training, and $f$'s output is used.
- **MoCo** (He et al., 2020) keeps a *queue* of negatives from previous batches, encoded by a slowly
  moving (momentum) copy of the encoder, so that small batches suffice.
- **BYOL** and **DINO** (Grill et al., 2020; Caron et al., 2021) drop negatives altogether: a *student*
  network predicts the output of a *teacher* that is an exponential moving average of the student, with
  centring and sharpening (DINO) to prevent collapse. DINO with a ViT backbone learns attention maps that
  segment objects without any labels (compare Section 5.3), and its linear probe reaches about 80 % on
  ImageNet.
- **CLIP** (Radford et al., 2021) applies the contrastive idea *across modalities*: an image encoder and a
  text encoder are trained on 400 million image-caption pairs so that matching pairs are similar. The
  result can classify images **zero-shot**: embed the prompts "a photo of a cat", "a photo of a dog", ...
  and pick the most similar one, with no training on the target classes at all. CLIP embeddings also
  condition text-to-image diffusion models (Notebook 12 · GANs and Diffusion Models).
- **Masked image modelling** (MAE, He et al., 2022) is the other big family: hide 75 % of the patches of
  a ViT's input and reconstruct them, the visual analogue of BERT's masked language modelling
  (Notebook 09 · Attention and Transformers).

### 7.2 The NT-Xent loss

SimCLR's loss is the **normalised-temperature cross-entropy** (NT-Xent). For a batch of $N$ images
with two views each, let $z_1, \dots, z_{2N}$ be the $\ell_2$-normalised embeddings,
$\mathrm{sim}(u, v) = u^\top v / (\|u\|\|v\|)$ the cosine similarity and $\tau$ a temperature. For a
positive pair $(i, j)$,

$$
\ell_{i,j} = -\log \frac{\exp(\mathrm{sim}(z_i, z_j)/\tau)}{\sum_{k=1,\,k \ne i}^{2N} \exp(\mathrm{sim}(z_i, z_k)/\tau)},
$$

and the loss is the average over all $2N$ ordered positive pairs. This is a softmax cross-entropy in which
the "classes" are the other $2N - 1$ views and the correct class is the partner view: exactly
`F.cross_entropy` on the similarity matrix with the diagonal masked out. With random embeddings every
view is equally similar to every other and the loss is $\log(2N - 1)$; the better the encoder separates
images, the lower it gets. The temperature ($\tau = 0.1$–$0.5$) controls how hard the negatives are
weighted. The implementation is six lines.

In [ ]:
def nt_xent(z1, z2, temperature=0.5):
    """NT-Xent (SimCLR) loss for two batches of embeddings of matching views, each (N, d)."""
    N = z1.shape[0]
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)                        # (2N, d), unit norm
    sim = (z @ z.T) / temperature                                              # scaled cosine similarities
    sim = sim.masked_fill(torch.eye(2 * N, dtype=torch.bool, device=z.device), float("-inf"))   # no self-pairs
    targets = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)]).to(z.device)               # i <-> i + N
    return F.cross_entropy(sim, targets)


N_PAIRS = 16
z_random = torch.randn(N_PAIRS, 64)
print(f"random embeddings, identical views : {nt_xent(z_random, z_random).item():.3f}")
print(f"random embeddings, unrelated views : {nt_xent(z_random, torch.randn(N_PAIRS, 64)).item():.3f}"
      f"   (log(2N - 1) = {math.log(2 * N_PAIRS - 1):.3f})")
print(f"same, temperature 0.1              : {nt_xent(z_random, z_random, temperature=0.1).item():.3f}")

Unrelated views give $\log(2N - 1) \approx 3.4$ as predicted. Identical views give a clearly lower loss,
but at $\tau = 0.5$ it is still around 1.6: the positive pair has cosine similarity 1 and the negatives
about 0, so after division by $\tau$ the positive logit is only 2 above the 30 negative ones, and
$-\log\big(e^2 / (e^2 + 30)\big) \approx 1.6$. At $\tau = 0.1$ the same similarities are 10 apart and the
loss almost vanishes: a lower temperature makes the same similarity differences count for more (a sharper
softmax), which is why SimCLR trains with $\tau$ around 0.1–0.5 rather than 1. We do not train SimCLR here
(it needs hundreds of epochs on large batches to pay off), but we can run the *evaluation* that matters:
create SimCLR-style augmentation pairs from CIFAR images, embed them with the randomly initialised and the
ImageNet-pretrained ResNet-18, and compare the NT-Xent loss and the fraction of views whose nearest
neighbour is their own partner. A representation that is invariant to the augmentations scores well on
both, and that is what contrastive training optimises directly.

In [ ]:
contrastive_tf = v2.Compose([
    v2.RandomResizedCrop(FT_SIZE, scale=(0.3, 1.0), antialias=True),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    v2.RandomGrayscale(p=0.2),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
torch.manual_seed(SEED)
pair_imgs = cifar_test_x[:N_PAIRS]
view1 = torch.stack([contrastive_tf(im) for im in pair_imgs])
view2 = torch.stack([contrastive_tf(im) for im in pair_imgs])


def denorm_imagenet(x):
    mean, std = torch.tensor(IMAGENET_MEAN).view(3, 1, 1), torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return to_np((x.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0))


fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for i in range(8):
    axes[0, i].imshow(denorm_imagenet(view1[i])); axes[1, i].imshow(denorm_imagenet(view2[i]))
    axes[0, i].set_title(f"pair {i}: view 1", fontsize=8); axes[1, i].set_title("view 2", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()


@torch.no_grad()
def contrastive_report(backbone):
    backbone.eval()
    z1 = backbone(view1.to(device)).float()
    z2 = backbone(view2.to(device)).float()
    loss = nt_xent(z1, z2).item()
    z = F.normalize(torch.cat([z1, z2]), dim=1)
    sim = z @ z.T
    sim.fill_diagonal_(-2.0)
    partner = torch.cat([torch.arange(N_PAIRS, 2 * N_PAIRS), torch.arange(N_PAIRS)]).to(device)
    retrieval = (sim.argmax(dim=1) == partner).float().mean().item()
    return loss, retrieval, sim.cpu()


print(f"{'encoder':28s} {'NT-Xent':>8s} {'partner retrieval':>18s}   (chance loss {math.log(2 * N_PAIRS - 1):.2f}, chance retrieval {1 / (2 * N_PAIRS - 1):.2f})")
for name, bb in [("ResNet-18, random init", backbone_rand), ("ResNet-18, ImageNet", backbone_pre)]:
    loss, retrieval, sim = contrastive_report(bb)
    print(f"{name:28s} {loss:8.3f} {100 * retrieval:17.0f}%")

plt.figure(figsize=(5.2, 4.6))
plt.imshow(to_np(sim), cmap="viridis", vmin=-0.2, vmax=1); plt.colorbar(fraction=0.046)
plt.title("cosine similarity between the 32 views (ImageNet encoder)")
plt.xlabel("view index (0-15: view 1, 16-31: view 2)"); plt.ylabel("view index"); plt.grid(False)
plt.tight_layout(); plt.show()

The pretrained encoder has a much lower NT-Xent loss and retrieves the partner view far more often than
the random encoder, whose features change drastically under cropping and colour jitter. In the similarity
matrix the two off-diagonal bands ($i$ versus $i + 16$) are the positive pairs; a perfect encoder would
light up exactly those. Contrastive pretraining pushes a randomly initialised network towards the
pretrained network's behaviour on this test, *using only unlabelled images*, which is why it produces
transferable features. Notebook 11 · Autoencoders and Variational Autoencoders picks up representation
learning from the generative side.

## Exercises

### Exercise 1 (warm-up): a different backbone

Repeat the linear-probe experiment of Section 3.2 with MobileNetV3-Small
(`torchvision.models.mobilenet_v3_small`, weights `MobileNet_V3_Small_Weights.DEFAULT`, 2.5 M parameters).
Its penultimate features are 576-dimensional and are obtained by replacing `model.classifier` with
`nn.Identity()`. Guard the weight download as in Section 2. Compare feature dimension, backbone size,
feature-extraction time and probe accuracy with ResNet-18. Is the smaller network a better choice for
this task?

**Solution to Exercise 1**

MobileNetV3 uses the same ImageNet normalisation, so the loaders of Section 3.1 can be reused unchanged;
only the headless model differs.

In [ ]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

try:
    mnet = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
    print("MobileNetV3-Small ImageNet weights loaded.")
except Exception as e:
    print(f"WARNING: could not obtain MobileNetV3 weights ({type(e).__name__}); using random init.")
    mnet = mobilenet_v3_small(weights=None)
mnet.classifier = nn.Identity()                        # features -> avgpool -> flatten -> (576,)
mnet = mnet.to(device)

t0 = time.time()
m_train_feats, m_train_labels = extract_features(mnet, ft_train_loader)     # labels from the same (shuffled) pass
m_test_feats, m_test_labels = extract_features(mnet, ft_test_loader)
t_mnet = time.time() - t0
_, acc_mnet = train_linear_probe(m_train_feats, m_train_labels, m_test_feats, m_test_labels)

print(f"\n{'backbone':20s} {'params':>10s} {'feat dim':>9s} {'extract s':>10s} {'probe acc':>10s}")
print(f"{'ResNet-18':20s} {count_params(backbone_pre):>10,} {train_feats.shape[1]:>9d} {t_extract:>10.1f} {100 * acc_probe:>9.1f}%")
print(f"{'MobileNetV3-Small':20s} {count_params(mnet):>10,} {m_train_feats.shape[1]:>9d} {t_mnet:>10.1f} {100 * acc_mnet:>9.1f}%")

MobileNetV3-Small has 2.5 M parameters against ResNet-18's 11.7 M, and most of them sit in the
576-1024-1000 classifier that we removed: the backbone that the table counts has under a million
parameters, less than a tenth of ResNet-18's, and it extracts features several times faster, at the
cost of a few points of probe accuracy (its ImageNet top-1 is 67.7 % against 69.8 %). For a
deployment on a phone or a CPU server that difference is often a good trade; for the best accuracy one
would move the other way, to ResNet-50, ConvNeXt or a pretrained ViT. The point of the `weights` API is
that swapping the backbone is a one-line change.

### Exercise 2: Grad-CAM++

Grad-CAM++ (Chattopadhyay et al., 2018) replaces the global average of the gradients by a weighted
average that emphasises pixels with large positive gradients, which helps when the class appears several
times in the image or covers only a small part of it. With $g^k_{ij} = \partial y^c / \partial A^k_{ij}$, the
weights are

$$
\alpha^k_{ij} = \frac{(g^k_{ij})^2}{2\,(g^k_{ij})^2 + \sum_{a,b} A^k_{ab}\,(g^k_{ij})^3}, \qquad
w_k^c = \sum_{i,j} \alpha^k_{ij}\; \mathrm{ReLU}(g^k_{ij}),
$$

and the map is $\mathrm{ReLU}(\sum_k w_k^c A^k)$ as before. Implement it as a subclass of `GradCAM` that
overrides `channel_weights`, and compare the two maps on the sample images.

**Solution to Exercise 2**

Only the weight computation changes; the hooks, backward pass, upsampling and normalisation are inherited.
Positions where the denominator is zero (no gradient at all) get $\alpha = 0$.

In [ ]:
class GradCAMpp(GradCAM):
    """Grad-CAM++: pixel-wise weighted gradients instead of their global average."""

    def channel_weights(self):
        g, A = self.gradients, self.activations
        g2, g3 = g ** 2, g ** 3
        denom = 2 * g2 + g3 * A.sum(dim=(2, 3), keepdim=True)          # sum over (a, b) of A only; g stays per pixel
        alpha = torch.where(denom != 0, g2 / torch.where(denom != 0, denom, torch.ones_like(denom)), torch.zeros_like(denom))
        return (alpha * F.relu(g)).sum(dim=(2, 3), keepdim=True)


fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for row, (name, img) in enumerate(sample_images.items()):
    x = preprocess(to_chw_uint8(img))[None].to(device)
    shown = to_np(display_tf(to_chw_uint8(img)).permute(1, 2, 0))
    axes[row, 0].imshow(shown); axes[row, 0].set_title(name)
    for col, (label, engine_cls) in enumerate([("Grad-CAM", GradCAM), ("Grad-CAM++", GradCAMpp)], start=1):
        engine = engine_cls(pretrained, pretrained.layer4)
        cam, logits, cls = engine(x)
        engine.remove()
        axes[row, col].imshow(shown); axes[row, col].imshow(to_np(cam[0]), cmap="jet", alpha=0.5)
        axes[row, col].set_title(f"{label}: {categories[cls[0].item()][:20]}")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

On these single-object images the two methods agree on the main region; Grad-CAM++ usually spreads the
heat more evenly over the whole object instead of concentrating on its most discriminative part, and it
is more robust when the object is small. The difference is larger on images with several instances of
the class, which is the case the method was designed for.

### Exercise 3: patch-size ablation for the ViT

Train the ViT of Section 5 with patch size 8 (16 tokens instead of 64) using the same budget, and compare
parameters, time per epoch and accuracy with the patch-4 model. Why is the larger patch so much cheaper,
and what does it cost?

**Solution to Exercise 3**

Patch 8 quarters the number of tokens. The $QK^\top$ and attention-times-$V$ products are quadratic in
the sequence length, so they get 16x cheaper; everything that is linear in the number of tokens (the
query/key/value projections and the MLP layers) gets 4x cheaper. The patch-embedding matrix grows (each
patch has $8 \cdot 8 \cdot 3 = 192$ values instead of 48) while the positional embedding shrinks, which is
why the parameter count barely changes.

In [ ]:
torch.manual_seed(SEED)
vit_p8 = ViT(32, 8, VIT_DIM, VIT_DEPTH, VIT_HEADS)
print(f"--- ViT patch 8: {count_params(vit_p8):,} parameters, {(32 // 8) ** 2} tokens")
hist_p8 = fit(vit_p8, vit_train_loader, vit_test_loader, EPOCHS_VIT, lr=1e-3, weight_decay=0.05,
              optimizer="adamw", warmup_frac=0.1)

hist_p4 = compare["ViT"][1]
print(f"\n{'model':14s} {'tokens':>7s} {'params':>10s} {'s/epoch':>8s} {'test acc':>9s}")
for name, model, hist, tokens in [("ViT patch 4", compare["ViT"][0], hist_p4, 64), ("ViT patch 8", vit_p8, hist_p8, 16)]:
    print(f"{name:14s} {tokens:>7d} {count_params(model):>10,} {np.mean(hist['epoch_time']):8.1f} {100 * hist['test_acc'][-1]:8.1f}%")

The patch-8 model trains two to three times faster per epoch but loses accuracy: with only 16 tokens of
$8 \times 8$ pixels, the image is represented very coarsely, and the linear patch embedding has to do the
work that the first convolutional layers of a CNN would do. On $32 \times 32$ images patch 4 is a
reasonable compromise; patch 2 (256 tokens) would be more accurate still but 16x more expensive in the
attention layers. For ImageNet-scale images the same trade-off leads to the standard patch size of 16 at
$224$ pixels (196 tokens).

### Exercise 4: dataset-level IoU for the U-Net

The metric of Section 6.3 averages the IoU *per image* and treats an absent class as a perfect score.
Benchmarks such as PASCAL VOC and Cityscapes instead report the **mean IoU** computed from a confusion
matrix accumulated over the *whole test set*: for each class, intersection = diagonal entry, union = row
sum + column sum - diagonal. Implement this, report per-class and mean IoU for the U-Net, and compare
with the per-image number. Why do they differ?

**Solution to Exercise 4**

The confusion matrix is accumulated with `torch.bincount` on the combined index `true * K + pred`, which
is much faster than a double loop. The dataset-level IoU pools all pixels of a class, so large objects
dominate, while the per-image average gives every image the same weight and rewards the model for
images where a class is absent.

In [ ]:
@torch.no_grad()
def segmentation_confusion(model, loader, num_classes=3):
    model.eval()
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for x, y in loader:
        pred = model(x.to(device)).argmax(dim=1).cpu()
        cm += torch.bincount(y.flatten() * num_classes + pred.flatten(), minlength=num_classes ** 2).view(num_classes, num_classes)
    return cm


cm = segmentation_confusion(unet, seg_test_loader)
inter = cm.diag().float()
union = cm.sum(dim=0).float() + cm.sum(dim=1).float() - inter
iou_per_class = inter / union
_, dice_img, iou_img = evaluate_segmentation(unet, seg_test_loader)
print("confusion matrix (rows: true, cols: predicted):\n", to_np(cm))
for c, name in enumerate(["background", "circle", "rectangle"]):
    print(f"IoU {name:11s}: {iou_per_class[c]:.3f}")
print(f"dataset-level mean IoU : {iou_per_class.mean():.3f}")
print(f"per-image mean IoU     : {iou_img:.3f}   (Section 6.3 convention)")
print(f"pixel accuracy         : {inter.sum() / cm.sum():.3f}   (dominated by the background)")

The background IoU is highest (it is the largest and easiest class), the shapes are lower, and the
dataset-level mean is typically a little below the per-image mean, because the per-image version gets a
free 1.0 whenever a class is absent from an image and its prediction. Pixel accuracy is higher than both
and says almost nothing about the shapes. When you read a segmentation paper, check which convention its
"mIoU" uses; the numbers are not interchangeable.

### Exercise 5 (challenge): mixup and CutMix for fine-tuning

Two augmentations that operate on *pairs* of training images are widely used for fine-tuning:

- **mixup** (Zhang et al., 2018): $\tilde x = \lambda x_i + (1 - \lambda) x_j$ with
  $\lambda \sim \mathrm{Beta}(\alpha, \alpha)$, and the loss
  $\lambda\, \ell(f(\tilde x), y_i) + (1 - \lambda)\, \ell(f(\tilde x), y_j)$;
- **CutMix** (Yun et al., 2019): paste a random rectangle of $x_j$ into $x_i$; $\lambda$ is the fraction
  of the image that remains from $x_i$, and the loss is mixed the same way.

Implement both as functions that take a batch `(x, y)` and return `(x_mixed, y_a, y_b, lam)`, write a
training loop that uses the mixed loss, show a few mixed images, and fine-tune the pretrained ResNet-18
with CutMix using the budget of Section 3.3. Compare with plain fine-tuning. When would you expect these
augmentations to help, and when to hurt?

**Solution to Exercise 5**

Both mixers pair each image with a random permutation of the batch. For CutMix the rectangle has area
$1 - \lambda$ and the aspect ratio of the image; $\lambda$ is recomputed from the clipped rectangle. The
training loop is `train_one_epoch` with the mixed loss; evaluation is unchanged.

In [ ]:
def mixup_batch(x, y, alpha=0.4):
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam


def cutmix_batch(x, y, alpha=1.0):
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    H, W = x.shape[-2:]
    cut = math.sqrt(1 - lam)
    ch, cw = int(H * cut), int(W * cut)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y0, y1 = int(np.clip(cy - ch // 2, 0, H)), int(np.clip(cy + ch // 2, 0, H))
    x0, x1 = int(np.clip(cx - cw // 2, 0, W)), int(np.clip(cx + cw // 2, 0, W))
    x_mixed = x.clone()
    x_mixed[:, :, y0:y1, x0:x1] = x[perm][:, :, y0:y1, x0:x1]
    lam = 1 - (y1 - y0) * (x1 - x0) / (H * W)                # actual fraction kept from x
    return x_mixed, y, y[perm], lam


def fit_mixed(model, train_loader, test_loader, epochs, mixer, lr=0.01, param_groups=None, weight_decay=5e-4):
    """Fine-tuning loop with a pairwise mixing augmentation and the corresponding mixed loss."""
    model.to(device)
    params = param_groups if param_groups is not None else [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay, nesterov=True)
    total, warm = epochs * len(train_loader), max(1, int(0.05 * epochs * len(train_loader)))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(total - warm, 1))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    hist = {"train_loss": [], "test_acc": []}
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); loss_sum, n = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            x, y_a, y_b, lam = mixer(x, y)
            with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
                logits = model(x)
                loss = lam * F.cross_entropy(logits, y_a) + (1 - lam) * F.cross_entropy(logits, y_b)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            loss_sum += loss.item() * x.size(0); n += x.size(0)
        _, te_acc = evaluate(model, test_loader)
        hist["train_loss"].append(loss_sum / n); hist["test_acc"].append(te_acc)
        print(f"epoch {epoch:2d}/{epochs} | train (mixed) loss {loss_sum / n:.4f} | test acc {te_acc:.3f} | {time.time() - t0:5.1f}s")
    return hist


xb, yb = next(iter(ft_train_loader))
torch.manual_seed(SEED); np.random.seed(SEED)
xm, ya, yb2, lam_m = mixup_batch(xb, yb)
xc, _, yb3, lam_c = cutmix_batch(xb, yb)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
axes[0].imshow(denorm_imagenet(xb[0])); axes[0].set_title(f"image A: {CLASSES[yb[0]]}")
axes[1].imshow(denorm_imagenet(xm[0])); axes[1].set_title(f"mixup, lam={lam_m:.2f} ({CLASSES[yb2[0]]})")
axes[2].imshow(denorm_imagenet(xb[1])); axes[2].set_title(f"image B: {CLASSES[yb[1]]}")
axes[3].imshow(denorm_imagenet(xc[1])); axes[3].set_title(f"CutMix, lam={lam_c:.2f} ({CLASSES[yb3[1]]})")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

torch.manual_seed(SEED); np.random.seed(SEED)
cm_model = resnet18_for_cifar(pretrained_init=True)
hist_cutmix = fit_mixed(cm_model, ft_train_loader, ft_test_loader, EPOCHS_FT, mixer=cutmix_batch, lr=0.01,
                        param_groups=discriminative_param_groups(cm_model, base_lr=0.01))
print(f"\nfine-tuning, plain  : {100 * results['fine-tuning (discriminative LR)'][0]:.1f}%")
print(f"fine-tuning, CutMix : {100 * hist_cutmix['test_acc'][-1]:.1f}%")

Mixing augmentations are regularisers: they make the training task harder, raise the training loss, and
pay off only when the model would otherwise overfit, that is with small datasets and long schedules. In
our short fine-tuning run they are roughly neutral or slightly harmful (in a single `FAST_RUN` epoch the
comparison is noise); with 10-20 epochs on a few thousand images CutMix typically adds a point or two and
also improves calibration. They are standard in modern ImageNet recipes (DeiT, ConvNeXt), where training
runs for 300 epochs, and much less useful for a quick linear probe.

## Summary

- Features learned on ImageNet are generic in the early layers and task-specific only in the last ones,
  which is why a pretrained backbone can be reused for almost any natural-image task with far less data
  and compute than training from scratch.
- torchvision's `weights=` API bundles checkpoints, the matching preprocessing (`weights.transforms()`)
  and metadata; always preprocess with the ImageNet statistics the weights were trained with, and guard
  downloads so that code runs offline.
- Feature extraction (a linear probe on frozen features) is fast and safe for small datasets; full
  fine-tuning with discriminative learning rates is the most accurate when data suffices; gradual
  unfreezing sits between them. Test-time augmentation adds a few tenths of a point for the price of a
  second forward pass per image.
- Grad-CAM weights the last convolutional feature maps by the pooled gradient of the class score,
  $L^c = \mathrm{ReLU}(\sum_k \alpha_k^c A^k)$, and gives class-discriminative heatmaps from one backward
  pass; it is coarse, shows only positive evidence and needs validation (Notebook 16 · Interpretability,
  Robustness and Uncertainty).
- A Vision Transformer embeds image patches as tokens, adds a class token and positional embeddings, and
  applies self-attention blocks; it has weak inductive biases and needs large-scale (pre)training, on
  small datasets a CNN wins.
- The U-Net's encoder-decoder with concatenation skips solves dense prediction; segmentation is
  evaluated with Dice and IoU, not pixel accuracy.
- Detectors predict a set of boxes via anchors (Faster R-CNN, YOLO) or object queries (DETR); IoU and
  non-maximum suppression are the two basic operations, both a few lines of code.
- Self-supervised methods (SimCLR/NT-Xent, MoCo, DINO, CLIP, MAE) learn transferable representations from
  unlabelled data by enforcing invariance to augmentations or by reconstruction; CLIP extends the idea to
  image-text pairs and enables zero-shot classification.

## Further reading

- Yosinski, J., Clune, J., Bengio, Y. and Lipson, H. (2014). How Transferable Are Features in Deep Neural Networks? *NeurIPS*.
- Kornblith, S., Shlens, J. and Le, Q. V. (2019). Do Better ImageNet Models Transfer Better? *CVPR*.
- Howard, J. and Ruder, S. (2018). Universal Language Model Fine-tuning for Text Classification. *ACL*. (Discriminative learning rates and gradual unfreezing)
- Selvaraju, R. R. et al. (2017). Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization. *ICCV*.
- Chattopadhyay, A., Sarkar, A., Howlader, P. and Balasubramanian, V. N. (2018). Grad-CAM++: Improved Visual Explanations for Deep Convolutional Networks. *WACV*.
- Adebayo, J. et al. (2018). Sanity Checks for Saliency Maps. *NeurIPS*.
- Dosovitskiy, A. et al. (2021). An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale. *ICLR*. (ViT)
- Touvron, H. et al. (2021). Training Data-Efficient Image Transformers and Distillation through Attention. *ICML*. (DeiT)
- Ronneberger, O., Fischer, P. and Brox, T. (2015). U-Net: Convolutional Networks for Biomedical Image Segmentation. *MICCAI*.
- Ren, S., He, K., Girshick, R. and Sun, J. (2015). Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks. *NeurIPS*.
- Redmon, J., Divvala, S., Girshick, R. and Farhadi, A. (2016). You Only Look Once: Unified, Real-Time Object Detection. *CVPR*.
- Carion, N. et al. (2020). End-to-End Object Detection with Transformers. *ECCV*. (DETR)
- Chen, T., Kornblith, S., Norouzi, M. and Hinton, G. (2020). A Simple Framework for Contrastive Learning of Visual Representations. *ICML*. (SimCLR)
- Caron, M. et al. (2021). Emerging Properties in Self-Supervised Vision Transformers. *ICCV*. (DINO)
- Radford, A. et al. (2021). Learning Transferable Visual Models from Natural Language Supervision. *ICML*. (CLIP)
- Yun, S. et al. (2019). CutMix: Regularization Strategy to Train Strong Classifiers with Localizable Features. *ICCV*; Zhang, H. et al. (2018). mixup: Beyond Empirical Risk Minimization. *ICLR*.
- torchvision documentation: models and pretrained weights, `transforms.v2`, `torchvision.ops`. https://docs.pytorch.org/vision/stable/